# Agent Platform 上での Kubeflow Pipeline の CI/CD

**学習目標:**
1. Agent Platform Pipelines を操作するためのカスタム Cloud Build ビルダーを作成する方法を学ぶ
1. KFP のすべてのアーティファクトをビルドして push するための Cloud Build 構成ファイルを作成する方法を学ぶ
1. Kubeflow Pipeline の新しい実行をトリガーする Cloud Build GitHub トリガーを設定する方法を学ぶ

このラボでは、Agent Platform 上で Kubeflow Pipeline を自動的にビルド、デプロイ、実行する **Cloud Build** CI/CD ワークフローの作成手順を学習します。また、パイプラインのコードをホストしている **GitHub** リポジトリに新しいタグが適用されたときにワークフローを開始するトリガーを設定し、ワークフローを **GitHub** と統合します。

## 環境設定の構成

In [ ]:
PROJECT_ID = !(gcloud config get-value project)
PROJECT_ID = PROJECT_ID[0]
REGION = "us-central1"
ARTIFACT_STORE = f"gs://{PROJECT_ID}-kfp-artifact-store"
os.environ["REGION"] = REGION
os.environ["ARTIFACT_STORE"] = ARTIFACT_STORE

アーティファクト ストアが存在することを確認しましょう。

In [ ]:
!gcloud storage ls | grep ^{ARTIFACT_STORE}/$ || gcloud storage buckets create --location={REGION} {ARTIFACT_STORE}

また、このノートブックでは、[ウォークスルー ノートブック](https://github.com/GoogleCloudPlatform/asl-ml-immersion/blob/master/notebooks/kubeflow_pipelines/walkthrough/solutions/kfp_walkthrough_vertex.ipynb) で説明されている手順に従って、データセットがすでに作成され Google Cloud Storage に保存されていることを前提としています。

まだ実行していない場合は、パイプラインを実行する前に下のセルを実行してデータセットを作成してください。

In [ ]:
%%bash
gcloud storage cp gs://asl-public-data/data/covertype/training/dataset.csv $ARTIFACT_STORE/data/training/dataset.csv
gcloud storage cp gs://asl-public-data/data/covertype/validation/dataset.csv $ARTIFACT_STORE/data/validation/dataset.csv

## Agent Platform 用 KFP CLI ビルダーの作成

### 演習

下のセルで、次のような Dockerfile を記述してください。
* ベースイメージとして `python:3.12` を使用する
* Python パッケージの `kfp`（バージョン `2.15.2`）、`google-cloud-aiplatform`（バージョン `1.137.0`）、`fire` をインストールする
* エントリポイントとして `/bin/bash` を起動する

In [ ]:
%%writefile kfp-cli/Dockerfile

# TODO

### イメージをビルドしてプロジェクトの **Artifact Registry** に push する

In [ ]:
ARTIFACT_REGISTRY_DIR = "asl-artifact-repo"
KFP_CLI_IMAGE_NAME = "kfp-cli"
KFP_CLI_IMAGE_URI = f"us-docker.pkg.dev/{PROJECT_ID}/{ARTIFACT_REGISTRY_DIR}/{KFP_CLI_IMAGE_NAME}:latest"
KFP_CLI_IMAGE_URI

### 演習

下のセルで、`gcloud builds` を使用して `kfp-cli` Docker イメージをビルドし、プロジェクトの Artifact Registry に push してください。

In [ ]:
!gcloud builds # コマンドを完成させてください

## **Cloud Build** ワークフローの理解

### 演習

下のセルで、CI/CD ワークフローを記述し、**Cloud Build** 変数を使用して環境固有の設定を抽象化する方法を定義した `cloudbuild.yaml` ファイルを完成させます。

この CI/CD ワークフローは、[kfp_pipeline_lightweight.ipynb](../../pipelines/solutions/kfp_pipeline_lightweight.ipynb) ノートブックで手動で行った手順を自動化します。
1. トレーナー イメージのビルド
1. パイプラインのコンパイル
1. Agent Platform Pipeline 環境へのパイプラインのアップロードと実行
1. プロジェクトの **Artifact Registry** へのトレーナーの push
 

**Cloud Build** ワークフロー構成では、標準およびカスタムの [Cloud Build ビルダー](https://cloud.google.com/cloud-build/docs/cloud-builders) の両方を使用します。カスタム ビルダーは **KFP CLI** をカプセル化しています。

In [ ]:
%%writefile cloudbuild.yaml
# Copyright 2021 Google LLC

# Licensed under the Apache License, Version 2.0 (the "License"); you may not use this
# file except in compliance with the License. You may obtain a copy of the License at

# https://www.apache.org/licenses/LICENSE-2.0
    
# Unless required by applicable law or agreed to in writing, software 
# distributed under the License is distributed on an "AS IS"
# BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either 
# express or implied. See the License for the specific language governing 
# permissions and limitations under the License.

steps:
# トレーナー イメージをビルド
- name: # TODO
  args: # TODO
  dir:  # TODO


# コンパイル ステップで利用できるように、トレーナー イメージを push
- name: # TODO
  args: # TODO
  dir:  # TODO


# パイプラインをコンパイル
- name: 'us-docker.pkg.dev/$PROJECT_ID/asl-artifact-repo/kfp-cli'
  args:
  - '-c'
  - |
    kfp dsl compile # TODO
  env:
  - 'PIPELINE_ROOT=gs://$PROJECT_ID-kfp-artifact-store/pipeline'
  - 'PROJECT_ID=$PROJECT_ID'
  - 'REGION=$_REGION'
  - 'SERVING_CONTAINER_IMAGE_URI=us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-0:latest'
  - 'TRAINING_CONTAINER_IMAGE_URI=us-docker.pkg.dev/$PROJECT_ID/asl-artifact-repo/trainer_image_covertype:latest'
  - 'TRAINING_FILE_PATH=gs://$PROJECT_ID-kfp-artifact-store/data/training/dataset.csv'
  - 'VALIDATION_FILE_PATH=gs://$PROJECT_ID-kfp-artifact-store/data/validation/dataset.csv'
  dir: $_PIPELINE_FOLDER/pipeline

# パイプラインを実行
- name: 'us-docker.pkg.dev/$PROJECT_ID/asl-artifact-repo/kfp-cli'
  args:
  - '-c'
  - |
    python $_PIPELINE_FOLDER/kfp-cli/run_pipeline.py  # TODO

logsBucket: 'gs://$PROJECT_ID-cloudbuild'

# Artifact Registry にイメージを push
# TODO: プロジェクトの Docker レジストリに push するイメージを一覧表示します
images: # TODO


# パイプラインの実行がデフォルトのタイムアウト時間を超過するため、これが必要です
timeout: 10800s


ビルドログを保存するための GCS バケットを作成しましょう。

In [ ]:
BUCKET = PROJECT_ID + "-cicd-log"
os.environ["BUCKET"] = BUCKET

In [ ]:
%%bash

exists=$(gcloud storage ls --buckets | grep -w gs://${BUCKET}/)
if [ -n "$exists" ]; then
    echo -e "Bucket exists, let's not recreate it."
else
    echo "Creating a new GCS bucket."
    gcloud storage buckets create --location=${REGION} gs://${BUCKET}
    echo "Here are your current buckets:"
    gcloud storage ls
fi

## CI/CD 実行の手動トリガー

[gcloud builds submit コマンド](https://cloud.google.com/sdk/gcloud/reference/builds/submit) を使用して、**Cloud Build** の実行を手動でトリガーできます。

In [ ]:
SUBSTITUTIONS = f"_REGION={REGION},_PIPELINE_FOLDER=./"
SUBSTITUTIONS

In [ ]:
!gcloud builds submit . --config cloudbuild.yaml --substitutions {SUBSTITUTIONS}

**注:** CloudBuild から Agent Platform へのアクセスで問題が発生した場合は、**CloudShell** で次のコマンドを実行する必要がある場合があります。

```bash
PROJECT_ID=$(gcloud config get-value project)
PROJECT_NUMBER=$(gcloud projects list --filter="name=$PROJECT_ID" --format="value(PROJECT_NUMBER)")

gcloud projects add-iam-policy-binding $PROJECT_ID   --member serviceAccount:$PROJECT_NUMBER@cloudbuild.gserviceaccount.com   --role roles/editor
gcloud projects add-iam-policy-binding $PROJECT_ID     --member serviceAccount:$PROJECT_NUMBER-compute@developer.gserviceaccount.com     --role roles/storage.objectAdmin
```

## GitHub 統合の設定

## 演習

この演習では、[Cloud Build GitHub アプリ](https://github.com/marketplace/google-cloud-build) を使用して、CI/CD ワークフローを **GitHub** と統合します。
パイプラインのソースコードを管理する **GitHub** リポジトリに新しいタグが適用されたときに CI/CD ワークフローを開始するトリガーを設定します。ソース GitHub リポジトリとして、このリポジトリのフォークを使用します。

### ステップ 1: このリポジトリのフォークを作成する
[GitHub のドキュメント](https://help.github.com/en/github/getting-started-with-github/fork-a-repo) に従って、[このリポジトリ](https://github.com/GoogleCloudPlatform/asl-ml-immersion) をフォークします。

### ステップ 2: yaml ファイルの変更を反映してコミットを作成する

このリポジトリのフォークしたページに移動し、`asl-ml-immersion/notebooks/kubeflow_pipelines/cicd/labs/cloudbuild.yaml` ファイルを開きます。

[Edit] ボタンをクリックし、更新した yaml ファイルをページに直接コピーします。
![image](https://user-images.githubusercontent.com/6895245/158727133-e5d77f0c-354c-4b2b-a710-8209ee67571f.png)

[Commit changes] ボタンをクリックして、新しいコミットを作成します。
![image](https://user-images.githubusercontent.com/6895245/158727565-13b4981a-8bce-401b-8f1a-d09a33a163a8.png)

### ステップ 3: **Cloud Build** トリガーを作成する

前のステップで作成したフォークを Google Cloud プロジェクトに接続し、[GitHub アプリのトリガーの作成](https://cloud.google.com/cloud-build/docs/create-github-app-triggers) の記事の手順に従ってトリガーを作成します。**Edit trigger**（トリガーの編集）フォームで次の値を使用します。

|フィールド| 値 |
|-----|---------------------------------------------------------------------------|
|名前（Name）| [YOUR TRIGGER NAME]                                                       |
|説明（Description）| [YOUR TRIGGER DESCRIPTION]                                                |
|イベント（Event）| Tag                                                                       |
|ソース（Source）| [YOUR FORK]                                                               |
|タグ（正規表現）（Tag (regex)）| .*                                                                        |
|ビルド構成（Build Configuration）| Cloud Build 構成ファイル（yaml または json）                             |
|Cloud Build 構成ファイルの場所（Cloud Build configuration file location）| ./asl_mlops/notebooks/kubeflow_pipelines/cicd/labs/cloudbuild.yaml |
|サービス アカウント（Service account）| `<PROJECT NUMBER>-compute@developer.gserviceaccount.com`                  |


代入変数には次の値を使用します。

|変数|値|
|--------|-----|
|_REGION|us-central1|
|_PIPELINE_FOLDER|asl_mlops/notebooks/kubeflow_pipelines/cicd/labs

### ステップ 4: ビルドをトリガーする

自動ビルドを開始するには、[GitHub でリポジトリの新しいリリースを作成](https://help.github.com/en/github/administering-a-repository/creating-releases) します。または、`git` を使用してタグを適用することでビルドを開始することもできます。
```bash
git tag [TAG NAME]
git push origin --tags
```


上記のコマンドを実行すると、ビルドが自動的にトリガーされます。[こちら](https://console.cloud.google.com/cloud-build/builds) でビルドを確認できます。

Copyright 2021 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.